# 2. Logistic Regression
**Use case:** Financial Services — Consumer loan default prediction and credit scorecard

A lender approves thousands of unsecured personal loans a month and roughly 10-15% of them go bad within a year. Credit officers need a probability of default for every applicant to set approve/decline cut-offs, price the loan and reserve for losses. They have bureau data (credit score, delinquencies, inquiries, utilization) and application data (income, debt-to-income, employment). Success means a stable, well-calibrated score that rank-orders risk, passes model validation and fair-lending review, and produces clear adverse action reasons for every decline.

> 💡 A bank loan officer keeps a mental tally: plus points for a steady job, minus points for missed payments, plus for low debt. Logistic regression is that tally written down precisely, with the final total converted into a percentage chance of default.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## CONCEPT: Hand-code default odds for 20 borrowers
Write logistic regression from scratch with numpy gradient descent and turn two applicants into probabilities of default.

| 5W + H | |
|---|---|
| **Who** | A credit analyst who wants to see what is inside the PD model the risk team uses. |
| **What** | A two-feature probability-of-default (PD) model trained by gradient descent in about 20 lines. |
| **When** | At application time - the lender needs a PD before deciding to approve, decline or reprice. |
| **Where** | On a laptop with plain numpy; the same maths runs inside every loan origination system's scoring engine. |
| **Why** | Logistic regression is the backbone of credit scoring; seeing weights move and log-loss fall builds real intuition. |
| **How** | Standardise debt-to-income and missed payments, loop 200 gradient steps on log-loss, then score two new applicants. |

**🧠 ML concepts:** Sigmoid / logistic function, Log-loss (cross-entropy), Gradient descent, Feature standardisation, Odds ratios from coefficients  
**🐍 Python concepts:** List of tuples, numpy slicing [:, :2], Defining a function with def, Matrix multiplication with @, In-place update -=, Modulo % for periodic printing

In [ ]:
# CONCEPT: logistic regression from scratch - probability a loan applicant defaults
import numpy as np

# 🐍 PY: each tuple is one past borrower: (debt_to_income %, missed_payments_last_2y, defaulted 0/1)
borrowers = [
    (15, 0, 0), (20, 0, 0), (25, 1, 0), (18, 0, 0), (30, 0, 0), (22, 1, 0),
    (35, 2, 0), (28, 0, 0), (45, 3, 1), (50, 2, 1), (40, 4, 1), (55, 1, 1),
    (38, 3, 1), (60, 5, 1), (33, 2, 1), (42, 0, 0),
    (24, 2, 1), (47, 1, 0), (29, 3, 0), (36, 1, 1),
]
# 🐍 PY: np.array on a list of tuples gives a 2-D table; [:, :2] = all rows, first two columns
data = np.array(borrowers, dtype=float)
X, y = data[:, :2], data[:, 2]
# 🧠 ML: scaling features (z-score) makes gradient descent converge smoothly
mu, sd = X.mean(axis=0), X.std(axis=0)
Xs = (X - mu) / sd

# 🧠 ML: the sigmoid squashes any number into a 0-1 probability
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# 🐍 PY: np.zeros(2) creates [0., 0.] - one weight per feature; b is the intercept (bias)
w, b, lr = np.zeros(2), 0.0, 0.5
for step in range(201):
    # 🐍 PY: @ is matrix multiplication: each row's features times the weights, summed
    p = sigmoid(Xs @ w + b)
    # 🧠 ML: log-loss punishes confident wrong answers; we nudge weights downhill on it
    loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    w -= lr * (Xs.T @ (p - y)) / len(y)
    b -= lr * np.mean(p - y)
    # 🐍 PY: % is the remainder operator - print every 50th step only
    if step % 50 == 0:
        print(f"step {step:>3}  log-loss {loss:.3f}  weights DTI={w[0]:+.2f} missed={w[1]:+.2f}")

# 🧠 ML: positive weight = raises default risk; exp(weight) = odds multiplier per 1 std-dev increase
print(f"\nOdds multiplier per +1 std of DTI: {np.exp(w[0]):.2f}x, of missed payments: {np.exp(w[1]):.2f}x")

# Score two new applicants
new = np.array([[20, 0], [48, 3]], dtype=float)
probs = sigmoid(((new - mu) / sd) @ w + b)
# 🐍 PY: zip walks two sequences side by side
for (dti, missed), pd_ in zip(new, probs):
    verdict = "DECLINE" if pd_ > 0.5 else "APPROVE"
    print(f"DTI {dti:.0f}%, missed {missed:.0f} -> probability of default {pd_:.1%} -> {verdict}")

**Try this next:**
- Change lr to 0.05 and watch how many more steps the loss needs to settle.
- Add a third feature (e.g. years at address) to every tuple and to the weights.
- Change the decision cut-off from 0.5 to 0.2 - a typical bank is far more conservative.

## BUILD: Calibrated PD model on 4,000 loans
Fit scikit-learn logistic regression on realistic loan data, read odds ratios, check calibration by risk grade and set an approval policy.

| 5W + H | |
|---|---|
| **Who** | A credit risk modeller preparing a PD model for an unsecured consumer loan product. |
| **What** | A 12-month probability-of-default model with AUC, Gini, Brier score, odds ratios and a calibration table. |
| **When** | During model development, using a historical cohort whose 12-month outcomes are known. |
| **Where** | In the risk analytics environment; output PDs feed pricing, approval cut-offs and loss provisioning. |
| **Why** | PDs must both rank risk (Gini) and be accurate in level (calibration) because they drive pricing and capital. |
| **How** | Simulate 4,000 loans, stratified split, StandardScaler + LogisticRegression, then AUC/Gini/Brier, odds ratios, quintile calibration and a PD < 10% approval policy. |

**🧠 ML concepts:** Probability of default (PD), AUC and Gini coefficient, Brier score and calibration, Odds ratios from standardised coefficients, Avoiding leakage when scaling, Cut-off policy  
**🐍 Python concepts:** np.clip for realistic bounds, pandas column arithmetic, pd.qcut for buckets, groupby with named aggregation, Boolean masks and ~, Multi-line f-strings

In [ ]:
# BUILD: scikit-learn logistic regression for consumer loan probability of default (PD)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

rng = np.random.default_rng(42)
n = 4000
# 🐍 PY: np.clip keeps simulated values inside realistic bounds
df = pd.DataFrame({
    "credit_score": np.clip(rng.normal(690, 60, n), 450, 850).round(),
    "debt_to_income": np.clip(rng.normal(32, 10, n), 2, 70).round(1),
    "annual_income_k": np.clip(rng.lognormal(4.1, 0.45, n), 15, 400).round(1),
    "loan_amount_k": np.clip(rng.lognormal(2.6, 0.5, n), 1, 60).round(1),
    "delinq_2y": rng.poisson(0.3, n),
    "months_employed": rng.integers(0, 240, n),
})
logit = (-2.4 - 0.018 * (df.credit_score - 690) + 0.05 * (df.debt_to_income - 32)
         + 0.55 * df.delinq_2y - 0.004 * df.months_employed
         + 0.4 * np.log(df.loan_amount_k / df.annual_income_k * 10))
df["default_12m"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
print(f"Loans: {n}  12-month default rate: {df.default_12m.mean():.1%}")

# 🐍 PY: keeping column names in one list avoids typos when selecting the same columns again
features = ["credit_score", "debt_to_income", "annual_income_k", "loan_amount_k",
            "delinq_2y", "months_employed"]
X_tr, X_te, y_tr, y_te = train_test_split(df[features], df.default_12m, test_size=0.25,
                                          stratify=df.default_12m, random_state=42)
# 🧠 ML: fit the scaler on TRAIN only, then apply to test - otherwise test info leaks into training
scaler = StandardScaler().fit(X_tr)
model = LogisticRegression(max_iter=1000)
model.fit(scaler.transform(X_tr), y_tr)

# 🐍 PY: method chaining - transform the test rows, then predict, in one expression
pd_te = model.predict_proba(scaler.transform(X_te))[:, 1]
auc = roc_auc_score(y_te, pd_te)
# 🧠 ML: Gini = 2*AUC - 1 is the number credit-risk teams quote; Brier score checks calibration
print(f"AUC {auc:.3f} | Gini {2 * auc - 1:.3f} | Brier {brier_score_loss(y_te, pd_te):.3f}")

# 🧠 ML: coefficients on scaled data are comparable - sign = direction, size = strength
coefs = pd.Series(model.coef_[0], index=features).sort_values()
print("\nCoefficient (per 1 std) and odds ratio:")
for f, c in coefs.items():
    print(f"  {f:<16} {c:+.3f}   odds x{np.exp(c):.2f}")

# 🧠 ML: calibration check - average predicted PD vs actual default rate, by risk grade (quintile)
# 🐍 PY: pd.qcut splits values into equal-sized buckets; groupby then aggregates each bucket
res = pd.DataFrame({"pd": pd_te, "actual": y_te.values})
res["grade"] = pd.qcut(res.pd, 5, labels=["A-best", "B", "C", "D", "E-worst"])
table = res.groupby("grade", observed=True).agg(loans=("pd", "size"), predicted=("pd", "mean"),
                                                 actual=("actual", "mean"))
print("\nRisk grade calibration:")
print(table.round(3).to_string())

# Business interpretation: approve if PD < 10%
# 🐍 PY: comparing an array to a number gives a boolean mask; mask.mean() = share of True values
approve = pd_te < 0.10
print(f"\nPolicy PD < 10%: approve {approve.mean():.0%} of applicants, "
      f"bad rate among approved {y_te.values[approve].mean():.1%} "
      f"vs {y_te.values[~approve].mean():.1%} among declined")

**Try this next:**
- Add class_weight='balanced' and see what happens to calibration (predicted PDs inflate).
- Drop credit_score and see how much Gini falls.
- Change the policy to PD < 5% and compare approval rate vs bad rate.

## PRODUCTION: Points-based scorecard with adverse action
Build a production-style WoE scorecard: custom sklearn transformer, tuned via CV, scaled to points, with cut-off economics and adverse action reasons.

| 5W + H | |
|---|---|
| **Who** | Credit risk, model validation and compliance teams who must approve and explain every decline. |
| **What** | A saved scorecard pipeline producing a 3-digit score, score-band bad rates, a profit-based cut-off and top-3 decline reasons. |
| **When** | Built yearly or when performance degrades; used for every new application in real time. |
| **Where** | Model development platform; the joblib artifact is deployed to the loan origination decision engine. |
| **Why** | Regulators expect transparent, stable models; ECOA/Reg B requires specific adverse action reasons; points scorecards deliver both. |
| **How** | WoE binning class inside a Pipeline, GridSearchCV over bins and C, baseline comparison, PDO scaling to points, rank-ordering table, cut-off sweep, reason codes, joblib save. |

**🧠 ML concepts:** Weight of Evidence (WoE) and Information Value (IV), Regularisation strength C, Points-to-double-odds (PDO) scaling, Rank-ordering and score bands, Profit-based cut-off, Adverse action reasons and fair lending  
**🐍 Python concepts:** Writing a class that inherits from sklearn base classes, __init__ vs fit and trailing-underscore attributes, Dict comprehension, Returning multiple values from a function, pd.cut with labels, Series.nlargest

In [ ]:
# PRODUCTION: production credit scorecard - WoE binning, tuned logistic regression, points, adverse action
import os
import tempfile
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
FEATS = ["credit_score", "debt_to_income", "utilization_pct", "delinq_2y", "months_employed", "inquiries_6m"]
PDO, BASE_SCORE, BASE_ODDS = 20, 600, 8   # 600 points = 8:1 good:bad odds; +20 points doubles odds


def make_data(n=8000, seed=42):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "credit_score": np.clip(rng.normal(690, 65, n), 450, 850).round(),
        "debt_to_income": np.clip(rng.normal(33, 11, n), 1, 75).round(1),
        "utilization_pct": np.clip(rng.beta(1.1, 1.4, n) * 100, 0, 100).round(1),
        "delinq_2y": rng.poisson(0.35, n),
        "months_employed": rng.integers(0, 300, n),
        "inquiries_6m": rng.poisson(1.2, n),
    })
    # 🧠 ML: utilization has a U-shape risk (0% and >80% both riskier) - WoE bins capture this, raw LR cannot
    util_risk = 1.3 * (df.utilization_pct > 80) + 0.8 * (df.utilization_pct < 8)
    logit = (-2.6 - 0.017 * (df.credit_score - 690) + 0.045 * (df.debt_to_income - 33) + util_risk
             + 0.5 * df.delinq_2y - 0.003 * df.months_employed + 0.2 * df.inquiries_6m)
    df["bad"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    return df


# 🐍 PY: inheriting BaseEstimator + TransformerMixin makes our class work inside sklearn Pipelines
class WoEBinner(BaseEstimator, TransformerMixin):
    """Quantile-bin each feature and replace the bin with its Weight of Evidence."""

    # 🐍 PY: __init__ only stores settings; sklearn requires learning to happen in fit()
    def __init__(self, n_bins=5):
        self.n_bins = n_bins

    def fit(self, X, y):
        # 🐍 PY: attributes ending in _ are learned during fit (a scikit-learn convention)
        self.edges_, self.woe_, self.iv_ = {}, {}, {}
        y = np.asarray(y)
        for col in X.columns:
            edges = np.unique(np.quantile(X[col], np.linspace(0, 1, self.n_bins + 1)))
            edges[0], edges[-1] = -np.inf, np.inf
            b = np.digitize(X[col], edges[1:-1])
            goods, bads = np.bincount(b[y == 0], minlength=len(edges) - 1), np.bincount(b[y == 1], minlength=len(edges) - 1)
            # 🧠 ML: WoE = ln(%good / %bad) per bin; +0.5 smoothing avoids divide-by-zero on empty bins
            pg, pb = (goods + 0.5) / goods.sum(), (bads + 0.5) / bads.sum()
            woe = np.log(pg / pb)
            self.edges_[col], self.woe_[col] = edges, woe
            # 🧠 ML: Information Value sums how well the bins separate goods from bads (>0.1 = useful)
            self.iv_[col] = float(np.sum((pg - pb) * woe))
        return self

    def transform(self, X):
        # 🐍 PY: dict comprehension builds {column: WoE values} for every column in one line
        out = {c: self.woe_[c][np.digitize(X[c], self.edges_[c][1:-1])] for c in X.columns}
        return pd.DataFrame(out, index=X.index)


def to_points(pipe, X):
    """Convert each feature's WoE x coefficient into scorecard points (higher score = safer)."""
    # 🧠 ML: factor = points per unit of log-odds; offset anchors BASE_SCORE at BASE_ODDS
    factor = PDO / np.log(2)
    offset = BASE_SCORE - factor * np.log(BASE_ODDS)
    woe = pipe.named_steps["woe"].transform(X)
    lr = pipe.named_steps["lr"]
    # 🧠 ML: score = offset - factor * log-odds(bad); split the log-odds across features to get points
    pts = -factor * woe * lr.coef_[0]
    base_pts = offset - factor * lr.intercept_[0]
    return pts, base_pts


def main():
    df = make_data()
    X_tr, X_te, y_tr, y_te = train_test_split(df[FEATS], df.bad, test_size=0.3, stratify=df.bad, random_state=42)
    print(f"Applications: {len(df):,}  bad rate: {df.bad.mean():.1%}")

    pipe = Pipeline([("woe", WoEBinner()), ("lr", LogisticRegression(max_iter=1000))])
    # 🧠 ML: tune bin count and regularization C with 5-fold stratified cross-validation on AUC
    grid = {"woe__n_bins": [4, 6, 8], "lr__C": [0.01, 0.1, 1.0]}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    search = GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc", n_jobs=1).fit(X_tr, y_tr)
    model = search.best_estimator_
    print(f"Best params: {search.best_params_}  CV AUC {search.best_score_:.3f}")

    # Baseline: plain logistic regression on raw (unbinned) features
    # 🧠 ML: baseline on raw features shows what WoE binning adds (it captures non-linear U-shapes)
    raw = LogisticRegression(max_iter=2000).fit((X_tr - X_tr.mean()) / X_tr.std(), y_tr)
    auc_raw = roc_auc_score(y_te, raw.predict_proba((X_te - X_tr.mean()) / X_tr.std())[:, 1])
    auc_woe = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    print(f"Test AUC  raw LR {auc_raw:.3f} (Gini {2*auc_raw-1:.3f}) | WoE scorecard {auc_woe:.3f} (Gini {2*auc_woe-1:.3f})")

    binner = model.named_steps["woe"]
    print("\nFeature           IV     coef")
    for f, c in zip(FEATS, model.named_steps["lr"].coef_[0]):
        print(f"  {f:<15} {binner.iv_[f]:.3f}  {c:+.3f}")

    pts, base = to_points(model, X_te)
    score = (pts.sum(axis=1) + base).round()
    print(f"\nScore range on test set: {score.min():.0f} - {score.max():.0f}")
    # 🧠 ML: a scorecard should rank-order - bad rate must fall as the score band rises
    bands = pd.cut(score, [0, 560, 600, 640, 680, 2000], labels=["<560", "560-599", "600-639", "640-679", "680+"])
    tbl = pd.DataFrame({"band": bands, "bad": y_te.values}).groupby("band", observed=True)["bad"].agg(["size", "mean"])
    print(tbl.rename(columns={"size": "apps", "mean": "bad_rate"}).round(3).to_string())

    # Cut-off decision: expected profit = 0.9k margin per good loan, 6k loss per bad loan
    # 🧠 ML: the cut-off is a business decision - trade approval volume against expected losses
    print("\nCut-off | approve % | bad rate | expected profit $k")
    for cut in [560, 580, 600, 620, 640]:
        ok = score >= cut
        prof = 0.9 * ((y_te.values == 0) & ok).sum() - 6.0 * ((y_te.values == 1) & ok).sum()
        print(f"{cut:>7} | {ok.mean():>9.0%} | {y_te.values[ok].mean():>8.1%} | {prof:>12,.0f}")

    # 🧠 ML: adverse action reasons (ECOA / Reg B) = features where the applicant lost most points vs max
    max_pts = pts.max()
    i = int(np.argmin(score.values))
    # 🐍 PY: Series arithmetic aligns on column names; nlargest picks the biggest shortfalls
    shortfall = (max_pts - pts.iloc[i]).nlargest(3)
    print(f"\nDeclined applicant score {score.iloc[i]:.0f}. Top adverse action reasons:")
    for f, gap in shortfall.items():
        print(f"  - {f} (value {X_te.iloc[i][f]}), {gap:.0f} points below best bin")

    print("\nFair lending: no protected attributes used; run disparate-impact tests on proxies (ECOA/FHA).")
    print("Model risk: SR 11-7 documentation, independent validation, annual PSI/KS monitoring.")
    path = os.path.join(tempfile.mkdtemp(), "pd_scorecard.joblib")
    joblib.dump({"pipeline": model, "pdo": PDO, "base_score": BASE_SCORE}, path)
    print(f"Saved scorecard to {os.path.basename(path)}")


if __name__ == "__main__":
    main()

**Try this next:**
- Change PDO to 40 and BASE_ODDS to 20 - the ranking is identical, only the score scale changes.
- Set the loss per bad loan to 10 (k$) and see where the profit-maximising cut-off moves.
- Remove utilization_pct from FEATS and see how AUC and the adverse action reasons change.